# 03 - Dense Embedding và Sparse Representation (Hue Foods RAG MVP)

Notebook này minh họa luồng xử lý của Phase 3 trong Hue Foods RAG:
- Dense embedding bằng local model `intfloat/multilingual-e5-small` trên CPU;
- Sparse representation theo công thức TF-IDF deterministic trên tập từ vựng tiếng Việt.

Toàn bộ quá trình chạy hoàn toàn local, không cần OpenRouter API key hay kết nối mạng bên ngoài.


In [1]:
import math
import sys
import time
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
else:
    raise RuntimeError("Không tìm thấy thư mục backend/.")

from core.settings_loader import load_settings
from embedding.embedder import E5Embedder
from embedding.sparse_embedder import SparseEmbedder, tokenize
from ingestion.chunking.markdown_chunker import chunk_foods_markdown

settings = load_settings()
embedding_cfg = settings["embedding"]
print("model:", embedding_cfg["model"])
print("vector_size:", embedding_cfg["vector_size"])
print("device:", embedding_cfg["device"])
print("batch_size:", embedding_cfg["batch_size"])


model: intfloat/multilingual-e5-small
vector_size: 384
device: cpu
batch_size: 64


## Dense Embedding với Multilingual E5

Mô hình E5 yêu cầu phân biệt rõ hai vai trò:
- Tài liệu (documents/passages) được gắn tiền tố `passage: `;
- Câu truy vấn (queries) được gắn tiền tố `query: `.

`E5Embedder` tự động xử lý các tiền tố này và gọi trực tiếp `SentenceTransformer.encode()` với `batch_size=64` và `normalize_embeddings=True`.


In [2]:
embedder = E5Embedder(
    model_id=embedding_cfg["model"],
    dimension=embedding_cfg["vector_size"],
    device=embedding_cfg["device"],
    batch_size=embedding_cfg["batch_size"],
)

chunks = chunk_foods_markdown()
texts = [chunk["text"] for chunk in chunks]

started = time.perf_counter()
dense_vectors = embedder.embed_documents(texts)
elapsed_seconds = time.perf_counter() - started


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/home/minhhieu/hue_rag/backend/embedding/embedder.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  actual_dimension = model.get_sentence_embedding_dimension()


In [3]:
first_vector = dense_vectors[0]
first_norm = math.sqrt(sum(v * v for v in first_vector))

print("model_id:", embedder.model_id)
print("chunk_count:", len(chunks))
print("dense_shape:", f"{len(dense_vectors)} x {len(first_vector)}")
print("first_vector_norm:", round(first_norm, 4))
print("elapsed_seconds:", round(elapsed_seconds, 2))


model_id: intfloat/multilingual-e5-small
chunk_count: 572
dense_shape: 572 x 384
first_vector_norm: 1.0
elapsed_seconds: 36.19


### Ý nghĩa của kết quả Dense Embedding

- `dense_shape`: 572 vectors x 384 dimensions tương ứng với toàn bộ 572 canonical chunks của Hue Foods RAG.
- `first_vector_norm`: Chuẩn L2 xấp xỉ 1.0 xác nhận các vector đã được chuẩn hóa để tính Cosine Similarity bằng tích vô hướng (dot product).
- `elapsed_seconds`: Thời gian chạy thực tế trên CPU (dùng để quan sát hiệu năng, không phải điều kiện cứng).


In [4]:
sample = "Bún bò Huế"
query_vector = embedder.embed_query(sample)
document_vector = embedder.embed_documents([sample])[0]

cosine_similarity = sum(
    q * d for q, d in zip(query_vector, document_vector)
)
print("sample text:", repr(sample))
print("query vector dimension:", len(query_vector))
print("document vector dimension:", len(document_vector))
print("query/document cosine:", round(cosine_similarity, 4))


sample text: 'Bún bò Huế'
query vector dimension: 384
document vector dimension: 384
query/document cosine: 0.9401


## Sparse Representation với Deterministic TF-IDF

Sparse vector ánh xạ các token tiếng Việt thành chỉ số trong từ vựng và trọng số TF-IDF:

$$\text{idf}(t) = \ln\left(\frac{N + 1}{\text{df}(t) + 1}\right) + 1$$
$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)$$

Trong đó:
- $N$ là tổng số tài liệu;
- $\text{df}(t)$ là số tài liệu chứa từ $t$;
- $\text{tf}(t, d)$ là số lần từ $t$ xuất hiện trong tài liệu $d$.


In [5]:
sparse_corpus = ["bún bò huế", "cơm hến", "bún bò giò heo"]
sparse_embedder = SparseEmbedder().fit(sparse_corpus)

print("tokens:", tokenize(sparse_corpus[0]))
print("documents:", sparse_embedder.num_documents)
print("vocabulary size:", sparse_embedder.vocabulary_size)
print("encoded sample:", sparse_embedder.encode("bún bò bò huế"))


tokens: ['bún', 'bò', 'huế']
documents: 3
vocabulary size: 7
encoded sample: {'indices': [0, 1, 2], 'values': [1.2876820724517808, 2.5753641449035616, 1.6931471805599454]}


## Tính tương thích và Lexical Path trong MVP

- `SparseEmbedder` được duy trì trong Phase 3 để đảm bảo tương thích với schema lưu trữ của Phase 4.
- Trong luồng truy vấn (Phase 5), hệ thống sử dụng kết hợp dense candidates từ Qdrant và chấm điểm Python BM25 trên tập candidates đó.


## Handoff sang Phase 8 (Model Selection)

Các mô hình embedding từ xa (như OpenRouter Qwen3 Embedding) sẽ được đánh giá và so sánh toàn diện với baseline E5 trong Phase 8 trên cùng tập dữ liệu chuẩn và các metric đo lường thống nhất.
